# TRACE-ESUS figures from saved tables

This notebook is the presentation layer for TRACE-ESUS. It reads committed CSV artifacts only, never runs a simulator or fits a model, and writes the established PNG/PDF figure families. The importable package contributes visual constants and the save helper only.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from traceesus.plotting.style import FIGURE_SIZES_INCHES, PALETTE, configure_style, save_figure

configure_style()

ENDOTYPE = ROOT / "outputs_latent_endotyping"
TRANSPORT = ROOT / "outputs_transportability"
COUNTER = ROOT / "outputs_locked" / "outputs"
HF_GRID = ROOT / "outputs_hf_grid"
CONFOUNDING = ROOT / "outputs_confounding"

required = (
    ENDOTYPE / "endotype_discovery_summary.csv",
    ENDOTYPE / "endotype_discovery_example_patient.csv",
    TRANSPORT / "transportability_summary.csv",
    COUNTER / "counterfactual_summary.csv",
    HF_GRID / "latent_grid_raw.csv",
    CONFOUNDING / "confounding_sweep_raw.csv",
)
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f"Missing CSV inputs: {missing}"
print(f"Reading CSV artifacts from {ROOT}")

Reading CSV artifacts from /Users/neeamhayder/Documents/ESUS Endotyping with Causal Models


## Shared table-to-figure helpers

The helpers below only group, reshape, and display saved values. Confidence bands are taken directly from the summary CSVs.

In [2]:
def method_color(method):
    text = method.lower()
    if "oracle" in text:
        return PALETTE["oracle"]
    if "causal" in text or "counterfactual" in text:
        return PALETTE["causal"]
    if "adjusted" in text:
        return PALETTE["adjusted"]
    return PALETTE["associative"]


def line_panel(axis, table, *, x, metric, methods, title, ylabel):
    selected = table[table["metric"] == metric]
    for method in methods:
        rows = selected[selected["method"] == method].sort_values(x)
        color = method_color(method)
        axis.plot(rows[x], 100 * rows["mean"], marker="o", color=color, label=method)
        if {"ci95_low", "ci95_high"}.issubset(rows.columns):
            axis.fill_between(rows[x], 100 * rows["ci95_low"], 100 * rows["ci95_high"], color=color, alpha=0.14)
    axis.set(title=title, xlabel=x.replace("_", " ").title(), ylabel=ylabel)
    axis.grid(axis="y", color=PALETTE["grid"], alpha=0.7)


def paired_panels(table, *, x, methods, panels, size, title, output):
    figure, axes = plt.subplots(1, 2, figsize=size)
    for axis, (metric, panel_title, ylabel) in zip(axes, panels, strict=True):
        line_panel(axis, table, x=x, metric=metric, methods=methods, title=panel_title, ylabel=ylabel)
    axes[0].legend(frameon=False, fontsize=8)
    figure.suptitle(title, fontweight="bold")
    figure.tight_layout()
    save_figure(figure, output)

## Proposal-retained experiments

In [3]:
counter = pd.read_csv(COUNTER / "counterfactual_summary.csv")
counter_methods = counter["method"].drop_duplicates().tolist()
paired_panels(
    counter, x="renal_effect_sd", methods=counter_methods,
    panels=(("true_mechanism_accuracy", "A. True-mechanism ranking accuracy", "Patients correctly ranked (%)"),
            ("false_atrial_confounded_competing", "B. False atrial classification", "Confounded competing patients called atrial (%)")),
    size=FIGURE_SIZES_INCHES["counterfactual_primary"],
    title="Figure P1. Mechanism ranking under renal biomarker distortion",
    output=COUNTER / "figure_P1",
)

In [4]:
endotype = pd.read_csv(ENDOTYPE / "endotype_discovery_summary.csv")
endotype_methods = endotype["method"].drop_duplicates().tolist()
primary = [method for method in endotype_methods if "Associative latent" in method or "Biologically" in method]
paired_panels(
    endotype, x="renal_effect_sd", methods=primary,
    panels=(("accuracy", "True-mechanism ranking accuracy", "Patients correctly assigned (%)"),
            ("false_atrial_renal_competing", "False atrial classification", "Renal/competing patients called atrial (%)")),
    size=FIGURE_SIZES_INCHES["endotype_recovery"],
    title="Figure P1. Hidden-mechanism recovery under renal biomarker distortion",
    output=ENDOTYPE / "figure_P1_latent_recovery",
)

In [5]:
null = pd.read_csv(ENDOTYPE / "k1_null_summary.csv")
controls = [method for method in endotype_methods if method not in primary[:1]]
figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZES_INCHES["endotype_controls"])
line_panel(axes[0], endotype, x="renal_effect_sd", metric="accuracy", methods=controls,
           title="Recovery with renal adjustment and oracle reference", ylabel="Patients correctly assigned (%)")
axes[0].legend(frameon=False, fontsize=8)
axes[1].bar(np.arange(len(null)), 100 * null["false_k2_rate"], color=[method_color(m) for m in null["method"]])
axes[1].set_xticks(np.arange(len(null)), [m.replace(" latent class model", "\nLCM").replace("Biologically constrained latent SCM", "Causal\nSCM") for m in null["method"]])
axes[1].set(title="K=1 null behavior", ylabel="Null repeats selecting K=2 (%)")
axes[1].grid(axis="y", color=PALETTE["grid"], alpha=0.7)
figure.suptitle("Figure S1. Fairness and null-model controls", fontweight="bold")
figure.tight_layout()
save_figure(figure, ENDOTYPE / "figure_S1_controls")

In [6]:
patient = pd.read_csv(ENDOTYPE / "endotype_discovery_example_patient.csv")
positions = np.arange(len(patient))
figure, axis = plt.subplots(figsize=FIGURE_SIZES_INCHES["endotype_example_patient"])
axis.bar(positions - 0.17, patient["observed_value"], 0.34, label="Observed", color=PALETTE["associative"])
axis.bar(positions + 0.17, patient["renal_neutralized_value"], 0.34, label="Renal path removed", color="#DBEAFE", edgecolor=PALETTE["causal"])
axis.set_xticks(positions, patient["biomarker"])
axis.set_ylabel("Standardized biomarker value")
axis.legend(frameon=False)
figure.suptitle("Figure P2. Illustrative renal-impaired patient", fontweight="bold")
figure.tight_layout()
save_figure(figure, ENDOTYPE / "figure_P2_example_patient")

In [7]:
transport = pd.read_csv(TRANSPORT / "transportability_summary.csv")
transport_methods = transport["method"].drop_duplicates().tolist()
main_methods = [m for m in transport_methods if m.startswith(("Pooled", "Target-adjusted", "Modular"))]
paired_panels(
    transport, x="shift_index", methods=main_methods,
    panels=(("accuracy", "True-mechanism recovery", "Patients correctly assigned (%)"),
            ("false_atrial_renal_competing", "False atrial classification", "Renal/competing patients called atrial (%)")),
    size=FIGURE_SIZES_INCHES["transportability_primary"],
    title="Figure T1. Cross-hospital transport of hidden-mechanism recovery",
    output=TRANSPORT / "figure_T1_transportability",
)

In [8]:
degradation = pd.read_csv(TRANSPORT / "transport_degradation.csv")
figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZES_INCHES["transportability_controls"])
control_methods = [m for m in transport_methods if not m.startswith("Pooled")]
line_panel(axes[0], transport, x="shift_index", metric="accuracy", methods=control_methods,
           title="Target calibration and oracle reference", ylabel="Patients correctly assigned (%)")
axes[0].legend(frameon=False, fontsize=7)
axes[1].bar(np.arange(len(degradation)), 100 * degradation["mean_difference"], color=[method_color(m) for m in degradation["method"]])
axes[1].axhline(0, color=PALETTE["text"], linewidth=0.8)
axes[1].set_xticks(np.arange(len(degradation)), [m.split()[0] for m in degradation["method"]], rotation=25)
axes[1].set(title="Strong shift minus no shift", ylabel="Accuracy change (percentage points)")
figure.suptitle("Figure T2. Transport controls and degradation", fontweight="bold")
figure.tight_layout()
save_figure(figure, TRANSPORT / "figure_T2_transport_controls")

ablation = pd.read_csv(TRANSPORT / "ablations" / "ablation_accuracy_changes.csv")
figure, axis = plt.subplots(figsize=FIGURE_SIZES_INCHES["transportability_ablation"])
for method in main_methods:
    rows = ablation[ablation["method"] == method].sort_values("shift_index")
    axis.plot(rows["shift"], 100 * rows["mean_difference"], marker="o", label=method, color=method_color(method))
axis.axhline(0, color=PALETTE["text"], linewidth=0.8)
axis.set(ylabel="Accuracy change from no shift (percentage points)", title="Figure T3. One-factor target-hospital shift ablations")
axis.legend(frameon=False, fontsize=8)
figure.tight_layout()
save_figure(figure, TRANSPORT / "ablations" / "figure_T3_shift_ablations")

## Heart-failure grid and confounding extensions

In [9]:
scatter = pd.read_csv(HF_GRID / "cohort_scatter_sample.csv")
levels = sorted(scatter["renal_effect_sd"].unique())
figure, axes = plt.subplots(1, len(levels), figsize=(14, 4), sharex=True, sharey=True)
for axis, level in zip(axes, levels, strict=True):
    block = scatter[scatter["renal_effect_sd"] == level]
    for mechanism, color in (("atrial", PALETTE["causal"]), ("competing", PALETTE["associative"])):
        rows = block[block["true_mechanism"] == mechanism]
        axis.scatter(rows["nt_probnp"], rows["ptfv1"], s=5, alpha=0.18, color=color, label=mechanism)
    axis.set(title=f"Renal effect {level:g} SD", xlabel="NT-proBNP-like marker")
axes[0].set_ylabel("PTFV1-like marker")
axes[0].legend(frameon=False)
figure.suptitle("A. Marker plane by true mechanism", fontweight="bold")
figure.tight_layout()
save_figure(figure, HF_GRID / "figures" / "A_marker_plane", dpi=200)

In [10]:
drift = pd.read_csv(HF_GRID / "identity_drift_raw.csv")
means = drift.groupby("renal_effect_sd", as_index=False).mean(numeric_only=True)
figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(means["renal_effect_sd"], means["agreement_with_mechanism"], marker="o", label="Mechanism")
axes[0].plot(means["renal_effect_sd"], means["agreement_with_kidney_status"], marker="o", label="Kidney status")
axes[0].set(xlabel="Renal effect (SD)", ylabel="Label-invariant agreement", title="What the discovered class tracks")
axes[0].legend(frameon=False)
composition = [c for c in means.columns if c.startswith("composition__")]
axes[1].stackplot(means["renal_effect_sd"], *[means[c] for c in composition], labels=[c.replace("composition__", "") for c in composition], alpha=0.85)
axes[1].set(xlabel="Renal effect (SD)", ylabel="Class composition", title="Composition of the atrial-like class")
axes[1].legend(frameon=False, fontsize=7, loc="upper left")
figure.suptitle("B. Latent identity drift", fontweight="bold")
figure.tight_layout()
save_figure(figure, HF_GRID / "figures" / "B_identity_drift", dpi=200)

In [11]:
grid = pd.read_csv(HF_GRID / "latent_grid_raw.csv")
grid_methods = grid["method"].drop_duplicates().tolist()
heatmap_methods = [grid_methods[0], next(m for m in grid_methods if "biologically" in m)]
figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, method in zip(axes, heatmap_methods, strict=True):
    matrix = grid[grid["method"] == method].pivot_table(index="heart_failure_effect_sd", columns="renal_effect_sd", values="false_atrial__redundant", aggfunc="mean")
    image = axis.imshow(100 * matrix, origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=100)
    axis.set_xticks(range(len(matrix.columns)), matrix.columns)
    axis.set_yticks(range(len(matrix.index)), matrix.index)
    axis.set(title=method, xlabel="Renal effect (SD)", ylabel="Heart-failure effect (SD)")
figure.colorbar(image, ax=axes, label="False atrial calls (%)")
figure.suptitle("C. False atrial attribution in the redundant subgroup", fontweight="bold")
figure.subplots_adjust(top=0.82, wspace=0.25)
save_figure(figure, HF_GRID / "figures" / "C_latent_redundant_heatmap", dpi=200)

In [12]:
subgroups = ("uncomplicated", "renal_only", "heart_failure_only", "redundant")
strong = grid[(grid["renal_effect_sd"] == grid["renal_effect_sd"].max()) & (grid["heart_failure_effect_sd"] == grid["heart_failure_effect_sd"].max())]

def subgroup_bars(methods, metric, title, output):
    figure, axis = plt.subplots(figsize=(11, 5))
    width = 0.8 / len(methods)
    positions = np.arange(len(subgroups))
    for index, method in enumerate(methods):
        values = [100 * strong.loc[strong["method"] == method, f"{metric}__{group}"].mean() for group in subgroups]
        axis.bar(positions + index * width, values, width, label=method, color=method_color(method), alpha=0.85)
    axis.set_xticks(positions + width * (len(methods) - 1) / 2, [s.replace("_", " ") for s in subgroups])
    axis.set(ylabel="Rate (%)", title=title)
    axis.legend(frameon=False, fontsize=7)
    figure.tight_layout()
    save_figure(figure, output, dpi=200)

subgroup_bars([grid_methods[0], heatmap_methods[1]], "accuracy", "D. Recovery across prespecified nuisance profiles", HF_GRID / "figures" / "D_latent_subgroup_bars")
subgroup_bars(grid_methods[:4], "false_atrial", "D. False atrial attribution across four models", HF_GRID / "figures" / "D_latent_subgroup_bars_four_models")

In [13]:
renal_max = grid["renal_effect_sd"].max()
slice_data = grid[grid["renal_effect_sd"] == renal_max]
figure, axis = plt.subplots(figsize=(9.5, 5))
for method in grid_methods:
    rows = slice_data[slice_data["method"] == method].groupby("heart_failure_effect_sd", as_index=False)["accuracy"].mean()
    axis.plot(rows["heart_failure_effect_sd"], 100 * rows["accuracy"], marker="o", label=method, color=method_color(method))
axis.set(xlabel="Heart-failure effect (SD)", ylabel="Accuracy (%)", title="E. Recovery at the strongest renal effect")
axis.legend(frameon=False, fontsize=7)
figure.tight_layout()
save_figure(figure, HF_GRID / "figures" / "E_latent_hf_slice", dpi=200)

posterior_method = next(m for m in grid_methods if "biologically" in m)
query_method = next(m for m in grid_methods if "counterfactual query" in m)
paired = slice_data[slice_data["method"].isin((posterior_method, query_method))].pivot(index=["repeat", "heart_failure_effect_sd"], columns="method", values="accuracy").reset_index()
paired["difference"] = paired[query_method] - paired[posterior_method]
divergence = paired.groupby("heart_failure_effect_sd", as_index=False)["difference"].mean()
figure, axis = plt.subplots(figsize=(8.5, 4.8))
axis.plot(divergence["heart_failure_effect_sd"], 100 * divergence["difference"], marker="o", color=PALETTE["causal"])
axis.axhline(0, color=PALETTE["text"], linewidth=0.8)
axis.set(xlabel="Heart-failure effect (SD)", ylabel="Counterfactual minus posterior accuracy (points)", title="F. Query divergence")
figure.tight_layout()
save_figure(figure, HF_GRID / "figures" / "F_query_divergence", dpi=200)

In [14]:
paired_panels(
    grid.groupby(["renal_effect_sd", "method"], as_index=False).agg(mean=("accuracy", "mean")).assign(metric="accuracy", ci95_low=lambda x: x["mean"], ci95_high=lambda x: x["mean"]),
    x="renal_effect_sd", methods=grid_methods,
    panels=(("accuracy", "Overall recovery", "Accuracy (%)"), ("accuracy", "Recovery under increasing renal distortion", "Accuracy (%)")),
    size=(12, 5), title="Figure 2. Recovery across the renal axis", output=HF_GRID / "figures" / "figure2_recovery_lines",
)

renal_summary = grid.groupby(["renal_effect_sd", "method"], as_index=False).agg(mean=("false_atrial_renal_competing", "mean")).assign(metric="false_atrial_renal_competing", ci95_low=lambda x: x["mean"], ci95_high=lambda x: x["mean"])
renal_plot = pd.concat([renal_summary, renal_summary.assign(metric="accuracy")], ignore_index=True)
paired_panels(
    renal_plot, x="renal_effect_sd", methods=grid_methods,
    panels=(("false_atrial_renal_competing", "Renal-competing false atrial calls", "False atrial calls (%)"), ("accuracy", "Same renal-denominator view", "False atrial calls (%)")),
    size=(12, 5), title="Figure 2. Recovery with the renal-competing denominator", output=HF_GRID / "figures" / "figure2_recovery_lines_renal_denominator",
)

In [15]:
confounding = pd.read_csv(CONFOUNDING / "confounding_sweep_raw.csv")
confounding_methods = confounding["method"].drop_duplicates().tolist()
figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for axis, metric, title in zip(axes, ("accuracy", "false_atrial_renal_competing", "fitted_prior_atrial_given_renal"), ("Recovery", "False atrial attribution", "Fitted atrial prior if renal impaired"), strict=True):
    for method in confounding_methods:
        rows = confounding[confounding["method"] == method].groupby("atrial_probability_if_renal_impaired", as_index=False)[metric].mean()
        axis.plot(rows["atrial_probability_if_renal_impaired"], 100 * rows[metric], marker="o", label=method, color=method_color(method))
    axis.set(xlabel="True P(atrial | renal impaired)", ylabel="Percent", title=title)
axes[0].legend(frameon=False, fontsize=7)
figure.suptitle("Renal confounding sweep", fontweight="bold")
figure.tight_layout()
save_figure(figure, CONFOUNDING / "figures" / "confounding_sweep", dpi=200)

## Execution check

A successful run must reproduce all 17 established figure families in both PNG and PDF form.

In [16]:
expected_bases = (
    COUNTER / "figure_P1",
    ENDOTYPE / "figure_P1_latent_recovery", ENDOTYPE / "figure_S1_controls", ENDOTYPE / "figure_P2_example_patient",
    TRANSPORT / "figure_T1_transportability", TRANSPORT / "figure_T2_transport_controls", TRANSPORT / "ablations" / "figure_T3_shift_ablations",
    CONFOUNDING / "figures" / "confounding_sweep",
    HF_GRID / "figures" / "A_marker_plane", HF_GRID / "figures" / "B_identity_drift", HF_GRID / "figures" / "C_latent_redundant_heatmap",
    HF_GRID / "figures" / "D_latent_subgroup_bars", HF_GRID / "figures" / "D_latent_subgroup_bars_four_models", HF_GRID / "figures" / "E_latent_hf_slice",
    HF_GRID / "figures" / "F_query_divergence", HF_GRID / "figures" / "figure2_recovery_lines", HF_GRID / "figures" / "figure2_recovery_lines_renal_denominator",
)
produced = [base.with_suffix(suffix) for base in expected_bases for suffix in (".png", ".pdf")]
missing = [str(path) for path in produced if not path.is_file()]
assert not missing, f"Missing rendered figures: {missing}"
print(f"Reproduced {len(expected_bases)} figure families ({len(produced)} files) from CSVs only.")

Reproduced 17 figure families (34 files) from CSVs only.
